In [0]:
customers = [
    (1, "Arjun", 28, "Male", "Kochi"),
    (2, "Meera", 34, "Female", "Chennai"),
    (3, "Rahul", 45, "Male", "Bangalore"),
    (4, "Sneha", 25, "Female", "Kochi"),
    (5, "Vikram", 52, "Male", "Hyderabad"),
    (6, "Asha", 39, "Female", "Chennai"),
    (7, "Kiran", 31, "Male", "Delhi")
]
 
columns = ["customer_id", "name", "age", "gender", "city"]
df_customers = spark.createDataFrame(customers, columns)
df_customers.show()

+-----------+------+---+------+---------+
|customer_id|  name|age|gender|     city|
+-----------+------+---+------+---------+
|          1| Arjun| 28|  Male|    Kochi|
|          2| Meera| 34|Female|  Chennai|
|          3| Rahul| 45|  Male|Bangalore|
|          4| Sneha| 25|Female|    Kochi|
|          5|Vikram| 52|  Male|Hyderabad|
|          6|  Asha| 39|Female|  Chennai|
|          7| Kiran| 31|  Male|    Delhi|
+-----------+------+---+------+---------+



In [0]:
orders = [
    (101, 1, 201, 2, "Online"),
    (102, 2, 202, 1, "Offline"),
    (103, 1, 203, 3, "Online"),
    (104, 3, 201, 1, "Offline"),
    (105, 5, 204, 2, "Online"),
    (106, 8, 202, 1, "Offline"),  # Missing customer
    (107, 6, 203, 2, "Online")
]
 
columns = ["order_id", "customer_id", "product_id", "quantity", "order_type"]
df_orders = spark.createDataFrame(orders, columns)
df_orders.show()

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_type|
+--------+-----------+----------+--------+----------+
|     101|          1|       201|       2|    Online|
|     102|          2|       202|       1|   Offline|
|     103|          1|       203|       3|    Online|
|     104|          3|       201|       1|   Offline|
|     105|          5|       204|       2|    Online|
|     106|          8|       202|       1|   Offline|
|     107|          6|       203|       2|    Online|
+--------+-----------+----------+--------+----------+



In [0]:
products = [
    (201, "Laptop", "Electronics", 50000),
    (202, "Mobile", "Electronics", 20000),
    (203, "Shoes", "Fashion", 3000),
    (204, "Watch", "Fashion", 7000),
    (205, "TV", "Electronics", 60000)
]
 
columns = ["product_id", "product_name", "category", "price"]
df_products = spark.createDataFrame(products, columns)
df_products.show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       201|      Laptop|Electronics|50000|
|       202|      Mobile|Electronics|20000|
|       203|       Shoes|    Fashion| 3000|
|       204|       Watch|    Fashion| 7000|
|       205|          TV|Electronics|60000|
+----------+------------+-----------+-----+



In [0]:
payments = [
    (101, "Paid"),
    (102, "Paid"),
    (103, "Pending"),
    (104, "Paid"),
    (105, "Failed"),
    (107, "Paid")
]
 
columns = ["order_id", "payment_status"]
df_payments = spark.createDataFrame(payments, columns)
df_payments.show()

+--------+--------------+
|order_id|payment_status|
+--------+--------------+
|     101|          Paid|
|     102|          Paid|
|     103|       Pending|
|     104|          Paid|
|     105|        Failed|
|     107|          Paid|
+--------+--------------+



# Multi-Level Join
## Task 1
- Create final dataset by joining all 4 tables

In [0]:
df = df_customers.join(df_orders, "customer_id","left").join(df_products, "product_id","left").join(df_payments, "order_id", "left")
df.show()


+--------+----------+-----------+------+---+------+---------+--------+----------+------------+-----------+-----+--------------+
|order_id|product_id|customer_id|  name|age|gender|     city|quantity|order_type|product_name|   category|price|payment_status|
+--------+----------+-----------+------+---+------+---------+--------+----------+------------+-----------+-----+--------------+
|     103|       203|          1| Arjun| 28|  Male|    Kochi|       3|    Online|       Shoes|    Fashion| 3000|       Pending|
|     101|       201|          1| Arjun| 28|  Male|    Kochi|       2|    Online|      Laptop|Electronics|50000|          Paid|
|     102|       202|          2| Meera| 34|Female|  Chennai|       1|   Offline|      Mobile|Electronics|20000|          Paid|
|     104|       201|          3| Rahul| 45|  Male|Bangalore|       1|   Offline|      Laptop|Electronics|50000|          Paid|
|    NULL|      NULL|          4| Sneha| 25|Female|    Kochi|    NULL|      NULL|        NULL|       NUL

##  Task 2
- Show:
    - name
    - product_name
    - quantity
    - price
    - payment_status

In [0]:
df = df_customers.join(df_orders, "customer_id","inner").join(df_products, "product_id","inner").join(df_payments, "order_id", "inner")
df.select("name","product_name","quantity","price","payment_status").show()

+------+------------+--------+-----+--------------+
|  name|product_name|quantity|price|payment_status|
+------+------------+--------+-----+--------------+
| Arjun|      Laptop|       2|50000|          Paid|
| Meera|      Mobile|       1|20000|          Paid|
| Arjun|       Shoes|       3| 3000|       Pending|
| Rahul|      Laptop|       1|50000|          Paid|
|Vikram|       Watch|       2| 7000|        Failed|
|  Asha|       Shoes|       2| 3000|          Paid|
+------+------------+--------+-----+--------------+



# Transformations (Week 1 Concepts)
##  Task 3
- Create:
    - age_group
        - <30 → Young
        - 30–50 → Adult
        - 50 → Senior
 

In [0]:
from pyspark.sql.functions import when
df_customers.withColumn("age_group", when(df_customers.age < 30, "Young").when((df_customers.age >= 30) & (df_customers.age <= 50), "Adult").when(df_customers.age > 50, "Senior").otherwise("unknown")).display()

customer_id,name,age,gender,city,age_group
1,Arjun,28,Male,Kochi,Young
2,Meera,34,Female,Chennai,Adult
3,Rahul,45,Male,Bangalore,Adult
4,Sneha,25,Female,Kochi,Young
5,Vikram,52,Male,Hyderabad,Senior
6,Asha,39,Female,Chennai,Adult
7,Kiran,31,Male,Delhi,Adult


## Task 4
- Create:
    - revenue_category
      - <5000 → Low
      - 5000–20000 → Medium
      - 20000 → High

In [0]:
df = df_orders.join(df_products, "product_id","inner")
df = df.withColumn("total_amount",df.price*df.quantity)
df.withColumn("revenue_category",when(df.total_amount < 5000 , "Low").when((df.total_amount >= 5000) & (df.total_amount <= 10000), "Medium").when(df.total_amount > 10000, "High").otherwise("unknown")).show()

+----------+--------+-----------+--------+----------+------------+-----------+-----+------------+----------------+
|product_id|order_id|customer_id|quantity|order_type|product_name|   category|price|total_amount|revenue_category|
+----------+--------+-----------+--------+----------+------------+-----------+-----+------------+----------------+
|       201|     101|          1|       2|    Online|      Laptop|Electronics|50000|      100000|            High|
|       202|     102|          2|       1|   Offline|      Mobile|Electronics|20000|       20000|            High|
|       203|     103|          1|       3|    Online|       Shoes|    Fashion| 3000|        9000|          Medium|
|       201|     104|          3|       1|   Offline|      Laptop|Electronics|50000|       50000|            High|
|       204|     105|          5|       2|    Online|       Watch|    Fashion| 7000|       14000|            High|
|       202|     106|          8|       1|   Offline|      Mobile|Electronics|20

# Filtering + Joins
##  Task 5
- Show:
    - only Online orders

****

In [0]:
df_orders.filter(df_orders.order_type == "Online").show()

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_type|
+--------+-----------+----------+--------+----------+
|     101|          1|       201|       2|    Online|
|     103|          1|       203|       3|    Online|
|     105|          5|       204|       2|    Online|
|     107|          6|       203|       2|    Online|
+--------+-----------+----------+--------+----------+



##  Task 6
- Show:
    - Paid orders only

In [0]:
df = df_orders.join(df_payments, "order_id", "inner")
df.filter(df.payment_status == "Paid").show()

+--------+-----------+----------+--------+----------+--------------+
|order_id|customer_id|product_id|quantity|order_type|payment_status|
+--------+-----------+----------+--------+----------+--------------+
|     101|          1|       201|       2|    Online|          Paid|
|     102|          2|       202|       1|   Offline|          Paid|
|     104|          3|       201|       1|   Offline|          Paid|
|     107|          6|       203|       2|    Online|          Paid|
+--------+-----------+----------+--------+----------+--------------+



## Task 7
- Show:
    - customers from Kochi who placed orders

In [0]:
from pyspark.sql.functions import when
df_customers.join(df_orders,"customer_id","inner").filter(df_customers.city == "Kochi").show()

+-----------+-----+---+------+-----+--------+----------+--------+----------+
|customer_id| name|age|gender| city|order_id|product_id|quantity|order_type|
+-----------+-----+---+------+-----+--------+----------+--------+----------+
|          1|Arjun| 28|  Male|Kochi|     101|       201|       2|    Online|
|          1|Arjun| 28|  Male|Kochi|     103|       203|       3|    Online|
+-----------+-----+---+------+-----+--------+----------+--------+----------+



# Aggregations
##  Task 8
- Total revenue per city

In [0]:
from pyspark.sql.functions import sum

df = df_customers.join(df_orders, "customer_id","inner").join(df_products, "product_id","inner")
df = df.withColumn("total_amount",df.price*df.quantity)
df.groupBy("city").agg(sum("total_amount").alias("revenue")).show()

+---------+-------+
|     city|revenue|
+---------+-------+
|    Kochi| 109000|
|  Chennai|  26000|
|Bangalore|  50000|
|Hyderabad|  14000|
+---------+-------+



##  Task 9
- Total revenue per category

In [0]:
from pyspark.sql.functions import sum

df = df_customers.join(df_orders, "customer_id","inner").join(df_products, "product_id","inner")
df = df.withColumn("total_amount",df.price*df.quantity)
df.groupBy("category").agg(sum("total_amount").alias("revenue")).show()

+-----------+-------+
|   category|revenue|
+-----------+-------+
|Electronics| 170000|
|    Fashion|  29000|
+-----------+-------+



## Task 10
- Count of orders by payment_status

In [0]:
from pyspark.sql.functions import count

df = df_orders.join(df_payments, "order_id", "inner")
df.groupBy("payment_status").agg(count("*")).show()

+--------------+--------+
|payment_status|count(1)|
+--------------+--------+
|          Paid|       4|
|       Pending|       1|
|        Failed|       1|
+--------------+--------+



# Business Analysis
##  Task 11
- Top customer by revenue

In [0]:
df = df_customers.join(df_orders, "customer_id","inner").join(df_products, "product_id","inner")
df = df.withColumn("total_amount",df.price*df.quantity)
df.groupBy("customer_id").agg(sum("total_amount").alias("revenue")).orderBy(desc("revenue")).show(1)

+-----------+-------+
|customer_id|revenue|
+-----------+-------+
|          1| 109000|
+-----------+-------+
only showing top 1 row


##  Task 12
- Top product by revenue

In [0]:
df = df_orders.join(df_products,"product_id","inner")
df = df.withColumn("total_amount",df.price*df.quantity)
df.groupBy("product_id").agg(sum("total_amount").alias("revenue")).orderBy(desc("revenue")).show(1)

+----------+-------+
|product_id|revenue|
+----------+-------+
|       201| 150000|
+----------+-------+
only showing top 1 row


## Task 13
- City with highest revenue

In [0]:
df = df_customers.join(df_orders,"customer_id","inner").join(df_products,"product_id","inner")
df = df.withColumn("total_amount",df.price*df.quantity)
df.groupBy("city").agg(sum("total_amount").alias("revenue")).orderBy(desc("revenue")).show(1)

+-----+-------+
| city|revenue|
+-----+-------+
|Kochi| 109000|
+-----+-------+
only showing top 1 row


# Data Quality 
##  Task 14
- Find:
    - Orders with missing customers

In [0]:
from pyspark.sql.functions import col
df = df_orders.join(df_customers,"customer_id","left").filter(col("name").isNull()).show()

+-----------+--------+----------+--------+----------+----+----+------+----+
|customer_id|order_id|product_id|quantity|order_type|name| age|gender|city|
+-----------+--------+----------+--------+----------+----+----+------+----+
|          8|     106|       202|       1|   Offline|NULL|NULL|  NULL|NULL|
+-----------+--------+----------+--------+----------+----+----+------+----+



## Task 15
- Find:
    - Orders with missing payment

In [0]:
df = df_orders.join(df_payments,"order_id","left")
df.filter(df.payment_status.isNull()).show()


+--------+-----------+----------+--------+----------+--------------+
|order_id|customer_id|product_id|quantity|order_type|payment_status|
+--------+-----------+----------+--------+----------+--------------+
|     106|          8|       202|       1|   Offline|          NULL|
+--------+-----------+----------+--------+----------+--------------+



# Challenge 
##  Task 16
- Find:
    - Top product in each city based on revenue

In [0]:
from pyspark.sql.functions import desc,max
df = df_customers.join(df_orders,"customer_id","inner").join(df_products,"product_id","inner")
df = df.withColumn("total_amount",df.price*df.quantity)

cities = [row["city"] for row in df.select("city").distinct().collect()]
for city in cities:
    citydf = df.filter(df.city == city)
    group = citydf.groupBy("product_id").agg(sum("total_amount").alias("revenue")).orderBy(desc("revenue"))
    # group.show()
    print(city, group.first()[0])


Kochi 201
Chennai 202
Bangalore 201
Hyderabad 204


In [0]:

t = df.groupBy("city","product_id").agg(sum("total_amount").alias("revenue"))
# t.show()
m = df.groupBy("city").agg(max("total_amount").alias("max_revenue"))
# m.show()
t.join(m,"city","inner").filter(t.revenue == m.max_revenue).select("city","product_id","revenue").show()

+---------+----------+-------+
|     city|product_id|revenue|
+---------+----------+-------+
|    Kochi|       201| 100000|
|  Chennai|       202|  20000|
|Bangalore|       201|  50000|
|Hyderabad|       204|  14000|
+---------+----------+-------+



##  Task 17
- Find:
    - Revenue contribution by age_group

In [0]:
df = df_customers.join(df_orders,"customer_id","inner").join(df_products,"product_id","inner")
df = df.withColumn("total_amount",df.price*df.quantity)
newdf = df.withColumn("age_group", when(df.age < 30, "<30").when((df.age >= 30) & (df.age <= 50), "30-50").when(df.age > 50, ">50").otherwise("unknown"))
# newdf.show()
newdf.groupBy("age_group").agg(sum("total_amount").alias("total_amount")).orderBy(desc("total_amount")).show()



+---------+------------+
|age_group|total_amount|
+---------+------------+
|      <30|      109000|
|    30-50|       76000|
|      >50|       14000|
+---------+------------+



#  Insight Questions 
- Q1 Which category is most profitable?

In [0]:
df_orders.join(df_products,"product_id","inner").groupBy("category").agg(sum(df_products.price*df_orders.quantity).alias("total_price")).orderBy(desc("total_price")).first()[0]


'Electronics'

- Q2 Which city should the company expand in?

In [0]:
df = df_customers.join(df_orders,"customer_id","inner").join(df_products,"product_id","inner")
df = df.withColumn("total_amount",df.price*df.quantity)
df.groupBy("city").agg(sum("total_amount").alias("total_revenue")).orderBy(desc("total_revenue")).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|    Kochi|       109000|
|Bangalore|        50000|
|  Chennai|        26000|
|Hyderabad|        14000|
+---------+-------------+



In [0]:
df = df_customers.join(df_orders,"customer_id","inner").join(df_products,"product_id","inner")
df = df.withColumn("total_amount",df.price*df.quantity)
df.groupBy("city","category").agg(sum("total_amount").alias("total_revenue")).orderBy(desc("total_revenue")).show()

+---------+-----------+-------------+
|     city|   category|total_revenue|
+---------+-----------+-------------+
|    Kochi|Electronics|       100000|
|Bangalore|Electronics|        50000|
|  Chennai|Electronics|        20000|
|Hyderabad|    Fashion|        14000|
|    Kochi|    Fashion|         9000|
|  Chennai|    Fashion|         6000|
+---------+-----------+-------------+



- Q3 Are online or offline orders more successful?

In [0]:
df = df_orders.join(df_products,"product_id","inner")
df = df.withColumn("total_amount",df.price*df.quantity)
df.groupBy("order_type").agg(sum("total_amount").alias("total_revenue")).orderBy(desc("total_revenue")).show(1)

+----------+-------------+
|order_type|total_revenue|
+----------+-------------+
|    Online|       129000|
+----------+-------------+
only showing top 1 row


- Q4 What data issues do you observe?

some of the details of the orders and payments of customers are missing from table.

In [0]:
# Data issues observed:
# - Orders with missing customers 
df_orders.join(df_customers, "customer_id", "left").filter(col("name").isNull()).display()

# - Orders with missing payment
df_orders.join(df_payments, "order_id", "left").filter(col("payment_status").isNull()).display()


customer_id,order_id,product_id,quantity,order_type,name,age,gender,city
8,106,202,1,Offline,null,null,null,null


order_id,customer_id,product_id,quantity,order_type,payment_status
106,8,202,1,Offline,null


In [0]:
df_orders.join(df_customers, "customer_id", "left").join(df_payments, "order_id", "left").filter((col("name").isNull()) | (col("payment_status").isNull())).show()

+--------+-----------+----------+--------+----------+----+----+------+----+--------------+
|order_id|customer_id|product_id|quantity|order_type|name| age|gender|city|payment_status|
+--------+-----------+----------+--------+----------+----+----+------+----+--------------+
|     106|          8|       202|       1|   Offline|NULL|NULL|  NULL|NULL|          NULL|
+--------+-----------+----------+--------+----------+----+----+------+----+--------------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, desc

In [0]:
Window.partitionBy("column").orderBy(desc("column"))